# Preparar a modelagem
Agora vamos construir a base para treinar um modelo de detecção de fraudes. Faremos isso com boas práticas:
1. Separar os dados em treino e teste.
2. Usar estratificação para preservar a proporção das classes.
3. Criar um pipeline para imputação, codificação das categorias e treinamento.
4. Avaliar o modelo com métricas adequadas ao desbalanceamento.
Um cuidado importante: a imputação anterior foi feita antes da divisão. Para uma avaliação rigorosa, vamos usar a base original, com os valores ausentes, e refazer a preparação dentro do pipeline, ajustando os imputadores apenas no conjunto de treino. Isso evita vazamento de dados.

In [1]:

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# Identificar a raiz do projeto
ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

# Carregar a base original, preservando os valores ausentes
CAMINHO_DADOS = (
    ROOT_DIR
    / "data"
    / "raw"
    / "sinistros_seguros_sinteticos.csv"
)

df = pd.read_csv(CAMINHO_DADOS)

print("Base carregada:", df.shape)
print("Arquivo:", CAMINHO_DADOS)
print("\nValores ausentes por coluna:")
display(df.isna().sum().to_frame("ausentes"))

print("\nDistribuição da variável-alvo:")
display(
    df["fraude_confirmada"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("percentual")
)


Base carregada: (20200, 14)
Arquivo: C:\Users\vanio\Documents\MEUS_PROJETOS\prevencao_fraude_seguro\data\raw\sinistros_seguros_sinteticos.csv

Valores ausentes por coluna:


,ausentes
id_sinistro,0
idade_segurado,607
tempo_cliente_anos,0
tipo_seguro,0
regiao,0
valor_sinistro,608
valor_apolice,0
sinistros_anteriores,0
dias_desde_contratacao,0
tipo_ocorrencia,609



Distribuição da variável-alvo:


,percentual
fraude_confirmada,
0,95.5
1,4.5


In [2]:

# 1. Remover registros totalmente duplicados
df_modelagem = df.drop_duplicates().copy()

print("Registros após remover duplicatas:", len(df_modelagem))
print(
    "Duplicatas restantes:",
    df_modelagem.duplicated().sum()
)

# 2. Separar a variável-alvo das variáveis explicativas
# O identificador não será usado como preditor
X = df_modelagem.drop(
    columns=["fraude_confirmada", "id_sinistro"]
)

y = df_modelagem["fraude_confirmada"]

print("\nDimensões de X:", X.shape)
print("Dimensões de y:", y.shape)

# 3. Dividir em treino e teste com estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTamanho do treino:", X_train.shape[0])
print("Tamanho do teste:", X_test.shape[0])

print("\nPercentual de fraude no treino:")
print(f"{y_train.mean() * 100:.2f}%")

print("Percentual de fraude no teste:")
print(f"{y_test.mean() * 100:.2f}%")


Registros após remover duplicatas: 20000
Duplicatas restantes: 0

Dimensões de X: (20000, 12)
Dimensões de y: (20000,)

Tamanho do treino: 16000
Tamanho do teste: 4000

Percentual de fraude no treino:
4.51%
Percentual de fraude no teste:
4.52%


# Criar o pipeline de pré-processamento
Agora vamos preparar os dados para um modelo de classificação. O pipeline vai:
- Preencher valores ausentes numéricos com a mediana.
- Preencher valores ausentes categóricos com "Não informado".
- Codificar categorias com OneHotEncoder.
- Manter as transformações dentro do pipeline, para evitar vazamento de dados.
Neste primeiro modelo, usaremos Regressão Logística como referência inicial. É um modelo útil para comparação e permite estimar probabilidades de fraude.

In [3]:

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

# Identificar tipos de variáveis
colunas_numericas = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

colunas_categoricas = X_train.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

print("Variáveis numéricas:", colunas_numericas)
print("\nVariáveis categóricas:", colunas_categoricas)

# Preparação das variáveis numéricas
pipeline_numerico = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="median")),
    ("escalonador", StandardScaler())
])

# Preparação das variáveis categóricas
pipeline_categorico = Pipeline(steps=[
    (
        "imputador",
        SimpleImputer(
            strategy="constant",
            fill_value="Não informado"
        )
    ),
    (
        "codificador",
        OneHotEncoder(handle_unknown="ignore")
    )
])

# Combinar os dois tipos de preparação
preprocessador = ColumnTransformer(transformers=[
    ("numericas", pipeline_numerico, colunas_numericas),
    ("categoricas", pipeline_categorico, colunas_categoricas)
])

# Criar o pipeline completo
modelo_logistico = Pipeline(steps=[
    ("preprocessamento", preprocessador),
    (
        "modelo",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

print("\nPipeline criado com sucesso.")


Variáveis numéricas: ['idade_segurado', 'tempo_cliente_anos', 'valor_sinistro', 'valor_apolice', 'sinistros_anteriores', 'dias_desde_contratacao', 'hora_ocorrencia', 'dias_para_comunicar']

Variáveis categóricas: ['tipo_seguro', 'regiao', 'tipo_ocorrencia', 'canal_aviso']

Pipeline criado com sucesso.


Por que usar class_weight="balanced"?
A base tem aproximadamente 4,5% de fraudes. Essa configuração ajusta os pesos das classes no treinamento, dando maior peso aos exemplos da classe minoritária.
Isso pode ajudar o modelo a identificar mais fraudes, mas também pode aumentar os falsos positivos. Vamos medir esse equilíbrio com as métricas de avaliação, em vez de presumir que o modelo ficou melhor.

## Treinamento do modelo

In [4]:

# Treinar o pipeline usando somente os dados de treino
modelo_logistico.fit(X_train, y_train)

print("Modelo treinado com sucesso!")

# Gerar previsões no conjunto de teste
y_pred = modelo_logistico.predict(X_test)

# Obter probabilidades estimadas de fraude
y_prob = modelo_logistico.predict_proba(X_test)[:, 1]

print("Previsões geradas:", len(y_pred))
print("Probabilidades geradas:", len(y_prob))
print("\nPrimeiras 10 probabilidades de fraude:")
print(np.round(y_prob[:10], 4))


Modelo treinado com sucesso!
Previsões geradas: 4000
Probabilidades geradas: 4000

Primeiras 10 probabilidades de fraude:
[0.5539 0.5192 0.441  0.5871 0.4818 0.4864 0.4845 0.4429 0.5644 0.4138]


In [5]:

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    accuracy_score
)

# Matriz de confusão
matriz = confusion_matrix(y_test, y_pred)

print("Matriz de confusão:")
display(
    pd.DataFrame(
        matriz,
        index=["Real: não fraude", "Real: fraude"],
        columns=["Previsto: não fraude", "Previsto: fraude"]
    )
)

# Métricas principais
metricas = {
    "Acurácia": accuracy_score(y_test, y_pred),
    "Precisão": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
    "PR-AUC (Average Precision)": average_precision_score(y_test, y_prob)
}

print("\nMétricas do modelo:")
display(
    pd.DataFrame.from_dict(
        metricas,
        orient="index",
        columns=["valor"]
    ).round(4)
)

print("\nRelatório de classificação:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Não fraude", "Fraude"],
        zero_division=0
    )
)


Matriz de confusão:


,Previsto: não fraude,Previsto: fraude
Real: não fraude,2087,1732
Real: fraude,84,97



Métricas do modelo:


,valor
Acurácia,0.5460
Precisão,0.0530
Recall,0.5359
F1-score,0.0965
ROC-AUC,0.5555
PR-AUC (Average Precision),0.0657



Relatório de classificação:
              precision    recall  f1-score   support

  Não fraude       0.96      0.55      0.70      3819
      Fraude       0.05      0.54      0.10       181

    accuracy                           0.55      4000
   macro avg       0.51      0.54      0.40      4000
weighted avg       0.92      0.55      0.67      4000



# Diagnóstico do primeiro modelo
Métrica	Resultado	Interpretação
Acurácia	54,60%	Muitas classificações incorretas
Precisão	5,30%	Poucos alertas correspondem a fraudes
Recall	53,59%	Identifica cerca de metade das fraudes
F1-score	9,65%	Baixo equilíbrio entre precisão e recall
ROC-AUC	0,5555	Discriminação pouco acima do acaso
PR-AUC	0,0657	Desempenho modesto; a referência de prevalência é aproximadamente 4,52%

### Próximo passo — Comparar com uma regressão logística sem pesos balanceados
Vamos treinar um segundo modelo, mantendo o mesmo pré-processamento e a mesma divisão treino/teste. A única mudança será retirar class_weight="balanced".

In [6]:

from sklearn.base import clone
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    accuracy_score
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Criar um segundo pipeline com o mesmo pré-processamento
modelo_logistico_sem_pesos = Pipeline(steps=[
    ("preprocessamento", clone(preprocessador)),
    (
        "modelo",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

# Treinar somente com o conjunto de treino
modelo_logistico_sem_pesos.fit(X_train, y_train)

# Prever classes e probabilidades no teste
y_pred_sem_pesos = modelo_logistico_sem_pesos.predict(X_test)
y_prob_sem_pesos = modelo_logistico_sem_pesos.predict_proba(X_test)[:, 1]

# Comparar as métricas
resultados_comparacao = pd.DataFrame({
    "Modelo balanceado": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0),
        roc_auc_score(y_test, y_prob),
        average_precision_score(y_test, y_prob)
    ],
    "Modelo sem pesos": [
        accuracy_score(y_test, y_pred_sem_pesos),
        precision_score(y_test, y_pred_sem_pesos, zero_division=0),
        recall_score(y_test, y_pred_sem_pesos, zero_division=0),
        f1_score(y_test, y_pred_sem_pesos, zero_division=0),
        roc_auc_score(y_test, y_prob_sem_pesos),
        average_precision_score(y_test, y_prob_sem_pesos)
    ]
}, index=[
    "Acurácia",
    "Precisão",
    "Recall",
    "F1-score",
    "ROC-AUC",
    "PR-AUC"
])

display(resultados_comparacao.round(4))


,Modelo balanceado,Modelo sem pesos
Acurácia,0.5460,0.9548
Precisão,0.0530,0.0000
Recall,0.5359,0.0000
F1-score,0.0965,0.0000
ROC-AUC,0.5555,0.5493
PR-AUC,0.0657,0.0647


Agora sabemos que nenhuma das duas configurações da Regressão Logística apresentou boa capacidade de discriminar fraudes.
O modelo sem pesos alcançou 95,48% de acurácia, mas não identificou nenhuma fraude. Isso acontece porque ele classificou todos os registros como não fraude. É um exemplo clássico de por que a acurácia isolada pode enganar.

In [7]:

# Analisar as taxas de fraude nas variáveis categóricas
for coluna in [
    "tipo_seguro",
    "regiao",
    "tipo_ocorrencia",
    "canal_aviso"
]:
    print(f"\nTaxa de fraude por {coluna}:")

    resumo = (
        df_modelagem
        .groupby(coluna, dropna=False)["fraude_confirmada"]
        .agg(["count", "mean"])
        .rename(columns={
            "count": "quantidade",
            "mean": "taxa_fraude"
        })
    )

    resumo["taxa_fraude_pct"] = (
        resumo["taxa_fraude"] * 100
    ).round(2)

    display(
        resumo.drop(columns="taxa_fraude")
        .sort_values("taxa_fraude_pct", ascending=False)
    )

# Verificar as taxas de fraude nas variáveis numéricas
colunas_numericas_eda = [
    "idade_segurado",
    "tempo_cliente_anos",
    "valor_sinistro",
    "valor_apolice",
    "sinistros_anteriores",
    "dias_desde_contratacao",
    "hora_ocorrencia",
    "dias_para_comunicar"
]

print("\nCorrelação das variáveis numéricas com a variável-alvo:")
display(
    df_modelagem[colunas_numericas_eda + ["fraude_confirmada"]]
    .corr(numeric_only=True)["fraude_confirmada"]
    .drop("fraude_confirmada")
    .sort_values(key=abs, ascending=False)
    .to_frame("correlacao")
    .round(4)
)



Taxa de fraude por tipo_seguro:


,quantidade,taxa_fraude_pct
tipo_seguro,,
Automóvel,12768,4.67
Vida,1990,4.37
Empresarial,1980,4.24
Residencial,3062,4.18
automóvel,200,4.00



Taxa de fraude por regiao:


,quantidade,taxa_fraude_pct
regiao,,
Centro-Oeste,3942,4.90
Sudeste,4050,4.47
Sul,3977,4.45
Nordeste,4010,4.44
Norte,4021,4.33



Taxa de fraude por tipo_ocorrencia:


,quantidade,taxa_fraude_pct
tipo_ocorrencia,,
Colisão,3894,4.78
Roubo,3749,4.56
Dano climático,3943,4.54
Outros,3926,4.46
Incêndio,3888,4.42
NaN,600,3.33



Taxa de fraude por canal_aviso:


,quantidade,taxa_fraude_pct
canal_aviso,,
NaN,600,5.17
Corretor,4769,4.76
Telefone,4822,4.56
Site,4900,4.39
Aplicativo,4909,4.28



Correlação das variáveis numéricas com a variável-alvo:


,correlacao
dias_para_comunicar,0.0326
valor_sinistro,0.0321
hora_ocorrencia,-0.0149
sinistros_anteriores,0.0101
idade_segurado,-0.0078
tempo_cliente_anos,-0.0064
valor_apolice,-0.0039
dias_desde_contratacao,-0.0003


O que os resultados mostram?
- Tipo de seguro: as taxas variam de 4,00% a 4,67%, uma diferença pequena. Também reaparece a inconsistência Automóvel/automóvel na base original.
- Região: Centro-Oeste apresenta 4,90%, enquanto Norte tem 4,33%. É uma diferença descritiva, não uma evidência de que a região cause fraude.
- Tipo de ocorrência: as taxas são relativamente próximas, entre 4,42% e 4,78% nas categorias conhecidas.
- Canal de aviso: os registros com canal ausente têm taxa de 5,17%, mas isso não significa que a ausência do canal cause fraude.
- Variáveis numéricas: todas as correlações lineares observadas estão próximas de zero.
Esses resultados não provam que seja impossível prever fraudes: podem existir interações e relações não lineares. Mas justificam testar outro algoritmo e comparar seu desempenho de forma rigorosa.

## Vamos testar uma Random Forest
Vamos testar um modelo capaz de aprender relações não lineares e interações entre variáveis. Manteremos a mesma divisão de treino e teste e o mesmo pré-processamento, para tornar a comparação justa.

In [8]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Criar o pipeline da Random Forest
modelo_rf = Pipeline(steps=[
    ("preprocessamento", clone(preprocessador)),
    (
        "modelo",
        RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

# Treinar
modelo_rf.fit(X_train, y_train)

# Avaliar no conjunto de teste
y_pred_rf = modelo_rf.predict(X_test)
y_prob_rf = modelo_rf.predict_proba(X_test)[:, 1]

# Comparar com os modelos anteriores
comparacao_modelos = pd.DataFrame({
    "Logística balanceada": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0),
        roc_auc_score(y_test, y_prob),
        average_precision_score(y_test, y_prob)
    ],
    "Logística sem pesos": [
        accuracy_score(y_test, y_pred_sem_pesos),
        precision_score(y_test, y_pred_sem_pesos, zero_division=0),
        recall_score(y_test, y_pred_sem_pesos, zero_division=0),
        f1_score(y_test, y_pred_sem_pesos, zero_division=0),
        roc_auc_score(y_test, y_prob_sem_pesos),
        average_precision_score(y_test, y_prob_sem_pesos)
    ],
    "Random Forest": [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf, zero_division=0),
        recall_score(y_test, y_pred_rf, zero_division=0),
        f1_score(y_test, y_pred_rf, zero_division=0),
        roc_auc_score(y_test, y_prob_rf),
        average_precision_score(y_test, y_prob_rf)
    ]
}, index=[
    "Acurácia",
    "Precisão",
    "Recall",
    "F1-score",
    "ROC-AUC",
    "PR-AUC"
])

display(comparacao_modelos.round(4))


,Logística balanceada,Logística sem pesos,Random Forest
Acurácia,0.5460,0.9548,0.9525
Precisão,0.0530,0.0000,0.0909
Recall,0.5359,0.0000,0.0055
F1-score,0.0965,0.0000,0.0104
ROC-AUC,0.5555,0.5493,0.5563
PR-AUC,0.0657,0.0647,0.0553


# Como interpretar:
- Logística sem pesos: apresenta acurácia alta porque praticamente classifica tudo como não fraude. Não identifica as fraudes neste teste.
- Random Forest: também apresenta acurácia alta, mas identifica pouquíssimas fraudes. Seu recall de 0,55% é muito baixo.
- Logística balanceada: identifica aproximadamente 53,6% das fraudes, mas sua precisão de 5,3% significa que a maioria dos alertas gerados é de falsos positivos.
- ROC-AUC: os valores próximos de 0,55 indicam capacidade de separação fraca, pouco acima de 0,50, que representa uma discriminação equivalente ao acaso.
A prevalência de fraude é de aproximadamente 4,52%. Portanto, a PR-AUC da logística balanceada (0,0657) está apenas modestamente acima dessa referência. O Random Forest, com 0,0553, não trouxe melhora nessa métrica.

from pathlib import Path

caminho_script = ROOT_DIR / "src" / "dataset.py"

print(caminho_script.read_text(encoding="utf-8"))

O ponto central: mesmo quando um sinistro apresenta fatores de risco, ele não é necessariamente rotulado como fraude. O resultado é sorteado aleatoriamente conforme a probabilidade calculada. Além disso, os efeitos dos fatores são relativamente pequenos.
Isso não significa que seja impossível aprender os padrões, mas explica por que a tarefa é difícil. Também há uma questão metodológica importante: os dados são artificiais, portanto não podemos interpretar esses padrões como evidências sobre fraudes reais em seguradoras.

In [10]:
# Diagnóstico dos fatores usados para gerar a fraude
df_diag = df_modelagem.copy()

df_diag["3 ou mais sinistros"] = (
    df_diag["sinistros_anteriores"] >= 3
)

df_diag["Contratação recente"] = (
    df_diag["dias_desde_contratacao"] < 60
)

df_diag["Sinistro de valor alto"] = (
    df_diag["valor_sinistro"]
    > df_diag["valor_sinistro"].quantile(0.90)
)

df_diag["Comunicação após 10 dias"] = (
    df_diag["dias_para_comunicar"] > 10
)

df_diag["Ocorrência até 4h"] = (
    df_diag["hora_ocorrencia"] <= 4
)

fatores = [
    "3 ou mais sinistros",
    "Contratação recente",
    "Sinistro de valor alto",
    "Comunicação após 10 dias",
    "Ocorrência até 4h",
]

# Taxa de fraude com e sem cada fator
for fator in fatores:
    resultado = (
        df_diag.groupby(fator, dropna=False)["fraude_confirmada"]
        .agg(["count", "mean"])
        .rename(columns={
            "count": "Quantidade",
            "mean": "Taxa de fraude"
        })
    )

    resultado["Taxa de fraude"] = (
        resultado["Taxa de fraude"] * 100
    ).round(2)

    print(f"\nFator: {fator}")
    display(resultado)


Fator: 3 ou mais sinistros


,Quantidade,Taxa de fraude
3 ou mais sinistros,,
False,19324,4.43
True,676,6.95



Fator: Contratação recente


,Quantidade,Taxa de fraude
Contratação recente,,
False,19646,4.42
True,354,9.60



Fator: Sinistro de valor alto


,Quantidade,Taxa de fraude
Sinistro de valor alto,,
False,18060,4.25
True,1940,7.01



Fator: Comunicação após 10 dias


,Quantidade,Taxa de fraude
Comunicação após 10 dias,,
False,7168,3.35
True,12832,5.17



Fator: Ocorrência até 4h


,Quantidade,Taxa de fraude
Ocorrência até 4h,,
False,15818,4.20
True,4182,5.71


temos evidências de que os cinco fatores realmente apresentam associação com a fraude simulada. Isso muda um pouco nosso diagnóstico: o gerador contém sinal preditivo, mas precisamos investigar por que os modelos não conseguem aproveitá-lo bem

## Próximo passo: verificar o modelo com um teste controlado
Minha recomendação é não alterar os dados nem aumentar artificialmente as métricas ainda. Primeiro, vamos verificar se a regressão logística consegue aprender diretamente os cinco fatores que criaram o alvo.
Isso nos ajudará a separar duas hipóteses:
- Se o modelo aprende bem usando os cinco fatores, o problema pode estar na representação dos dados ou na presença de variáveis irrelevantes.
- Se continua com desempenho fraco, precisamos investigar a força do sinal, o sorteio probabilístico e a forma como estamos avaliando o model

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report
)

fatores_modelo = [
    "sinistros_anteriores",
    "dias_desde_contratacao",
    "valor_sinistro",
    "dias_para_comunicar",
    "hora_ocorrencia",
]

modelo_diagnostico = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="median")),
    ("modelo", LogisticRegression(max_iter=1000))
])

modelo_diagnostico.fit(
    X_train[fatores_modelo],
    y_train
)

prob_diagnostico = modelo_diagnostico.predict_proba(
    X_test[fatores_modelo]
)[:, 1]

pred_diagnostico = modelo_diagnostico.predict(
    X_test[fatores_modelo]
)

print("ROC-AUC:", round(
    roc_auc_score(y_test, prob_diagnostico), 4
))

print("PR-AUC:", round(
    average_precision_score(y_test, prob_diagnostico), 4
))

print("\nRelatório de classificação:")
print(classification_report(
    y_test,
    pred_diagnostico,
    zero_division=0
))

ROC-AUC: 0.5625
PR-AUC: 0.0724

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98      3819
           1       0.00      0.00      0.00       181

    accuracy                           0.95      4000
   macro avg       0.48      0.50      0.49      4000
weighted avg       0.91      0.95      0.93      4000



O modelo com apenas os cinco fatores apresentou:
- ROC-AUC: 0,5625 — capacidade de discriminação fraca.
- PR-AUC: 0,0724 — acima da prevalência de fraude de 4,52%, mas ainda modesta.
- Recall de fraude: 0% no limiar padrão de 0,50 — o modelo não classificou nenhum caso de teste como fraude.
Isso não significa que o modelo não tenha aprendido absolutamente nada: as métricas de AUC usam as probabilidades, enquanto o relatório usa as classes previstas com o limiar padrão. O resultado sugere que as probabilidades ainda não são suficientemente altas para ultrapassar 0,50.
Há também uma hipótese que vale investigar: o gerador usa uma relação não linear, com indicadores como “três ou mais sinistros” e “contratação recente”, mas a regressão logística está recebendo os valores numéricos originais. Ela não representa diretamente esses limites.

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import pandas as pd

def criar_fatores(df, limite_valor):
    return pd.DataFrame({
        "sinistros_3_ou_mais": (
            df["sinistros_anteriores"] >= 3
        ).astype(int),

        "contratacao_recente": (
            df["dias_desde_contratacao"] < 60
        ).astype(int),

        "sinistro_valor_alto": (
            df["valor_sinistro"] > limite_valor
        ).astype(int),

        "comunicacao_tardia": (
            df["dias_para_comunicar"] > 10
        ).astype(int),

        "ocorrencia_madrugada": (
            df["hora_ocorrencia"] <= 4
        ).astype(int),
    })

# O limite do valor é calculado somente com os dados de treino
limite_valor = X_train["valor_sinistro"].quantile(0.90)

X_train_fatores = criar_fatores(X_train, limite_valor)
X_test_fatores = criar_fatores(X_test, limite_valor)

modelo_fatores = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="most_frequent")),
    ("modelo", LogisticRegression(max_iter=1000))
])

modelo_fatores.fit(X_train_fatores, y_train)

prob_fatores = modelo_fatores.predict_proba(
    X_test_fatores
)[:, 1]

pred_fatores = modelo_fatores.predict(X_test_fatores)

print("ROC-AUC:", round(
    roc_auc_score(y_test, prob_fatores), 4
))

print("PR-AUC:", round(
    average_precision_score(y_test, prob_fatores), 4
))

print("\nQuantidade de fraudes previstas:",
      int(pred_fatores.sum()))

print("\nCoeficientes aprendidos:")
display(pd.DataFrame({
    "Fator": X_train_fatores.columns,
    "Coeficiente": modelo_fatores.named_steps[
        "modelo"
    ].coef_[0]
}).sort_values("Coeficiente", ascending=False))

ROC-AUC: 0.5989
PR-AUC: 0.0632

Quantidade de fraudes previstas: 0

Coeficientes aprendidos:


,Fator,Coeficiente
1,contratacao_recente,0.762366
0,sinistros_3_ou_mais,0.576131
2,sinistro_valor_alto,0.559147
3,comunicacao_tardia,0.436373
4,ocorrencia_madrugada,0.289506


O que os resultados significam?
- ROC-AUC: 0,5989: melhor que os modelos anteriores, mas ainda indica discriminação fraca.
- PR-AUC: 0,0632: abaixo do resultado anterior de 0,0724. Portanto, não houve melhora em todas as métricas.
- Fraudes previstas: 0: com o limiar padrão de 0,50, o modelo não sinalizou nenhum caso.
- Coeficientes: todos positivos, coerentes com a direção dos fatores usados para gerar o rótulo. Isso é esperado porque estamos utilizando variáveis semelhantes às da fórmula original.
Há um detalhe importante: a taxa de fraude é baixa, aproximadamente 4,52%. Como o rótulo é sorteado probabilisticamente, mesmo conhecer os fatores geradores não permite prever cada fraude com certeza.

In [13]:
import pandas as pd
import numpy as np

print("Resumo das probabilidades previstas:")
display(pd.Series(prob_fatores).describe().round(4))

print("\nMaiores probabilidades previstas:")
print(
    np.sort(prob_fatores)[-10:][::-1].round(4)
)

print("\nPercentis das probabilidades:")
display(
    pd.Series(prob_fatores)
    .quantile([0.50, 0.75, 0.90, 0.95, 0.99, 1.00])
    .rename("Probabilidade")
    .to_frame()
    .round(4)
)

Resumo das probabilidades previstas:


count    4000.0000
mean        0.0450
std         0.0157
min         0.0284
25%         0.0376
50%         0.0433
75%         0.0487
max         0.1849
dtype: float64


Maiores probabilidades previstas:
[0.1849 0.1849 0.1584 0.1472 0.1451 0.1451 0.1235 0.1235 0.1235 0.1148]

Percentis das probabilidades:


,Probabilidade
0.50,0.0433
0.75,0.0487
0.90,0.0640
0.95,0.0734
0.99,0.0972
1.00,0.1849


agora entendemos por que o modelo não prevê nenhuma fraude: todas as probabilidades estimadas ficaram abaixo de 50%. A maior foi 18,49%, e a mediana foi 4,33%.
Isso é coerente com a baixa prevalência de fraude e com a regressão logística sem pesos. Uma probabilidade de 18,49%, porém, não significa que o caso seja fraude; significa que o modelo estima um risco maior em relação aos demais casos.

# avaliar um limiar de risco de 5%
Em vez de exigir probabilidade acima de 50%, podemos testar um limiar de 5% para entender quantos casos seriam sinalizados. Esse é um teste exploratório, não um limiar operacional recomendado.

In [14]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

limiar = 0.05
pred_limiar = (prob_fatores >= limiar).astype(int)

print(f"Limiar: {limiar:.0%}")
print("Casos sinalizados:", int(pred_limiar.sum()))
print("Precisão:", round(
    precision_score(y_test, pred_limiar, zero_division=0), 4
))
print("Recall:", round(
    recall_score(y_test, pred_limiar, zero_division=0), 4
))
print("F1-score:", round(
    f1_score(y_test, pred_limiar, zero_division=0), 4
))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, pred_limiar))

Limiar: 5%
Casos sinalizados: 909
Precisão: 0.0715
Recall: 0.3591
F1-score: 0.1193

Matriz de confusão:
[[2975  844]
 [ 116   65]]


Interpretação de negócio: a cada 100 casos sinalizados, aproximadamente 7 eram fraudes. A equipe de investigação precisaria analisar muitos casos legítimos para encontrar as fraudes detectadas.
O limiar de 5% é apenas exploratório. Não podemos considerá-lo o melhor limiar operacional sem avaliar os custos de investigação, as perdas evitadas e a capacidade da equipe.

# Estratégia  1 — comparar a regressão logística normal e a balanceada
Vamos usar uma parte dos dados de treino para ajustar os modelos e outra para validá-los. Assim, não precisamos escolher a estratégia usando novamente os resultados do teste.

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd

# Separar treino e validação a partir dos dados de treino existentes
X_dev, X_val, y_dev, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train
)

# Calcular o limite do valor de sinistro somente no treino
limite_valor_dev = X_dev["valor_sinistro"].quantile(0.90)

X_dev_fatores = criar_fatores(X_dev, limite_valor_dev)
X_val_fatores = criar_fatores(X_val, limite_valor_dev)

# Modelo padrão
modelo_padrao = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="most_frequent")),
    ("modelo", LogisticRegression(max_iter=1000))
])

# Modelo que dá mais peso à classe minoritária
modelo_balanceado = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="most_frequent")),
    ("modelo", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

modelos = {
    "Logística padrão": modelo_padrao,
    "Logística balanceada": modelo_balanceado
}

resultados = {}

for nome, modelo in modelos.items():
    modelo.fit(X_dev_fatores, y_dev)

    probabilidades = modelo.predict_proba(
        X_val_fatores
    )[:, 1]

    # Avaliação inicial com o limiar padrão de 0,50
    previsoes = (probabilidades >= 0.50).astype(int)

    resultados[nome] = {
        "ROC-AUC": roc_auc_score(y_val, probabilidades),
        "PR-AUC": average_precision_score(y_val, probabilidades),
        "Precisão": precision_score(
            y_val, previsoes, zero_division=0
        ),
        "Recall": recall_score(
            y_val, previsoes, zero_division=0
        ),
        "F1-score": f1_score(
            y_val, previsoes, zero_division=0
        ),
        "Fraudes previstas": int(previsoes.sum())
    }

comparacao_validacao = pd.DataFrame(resultados).T
display(comparacao_validacao.round(4))

,ROC-AUC,PR-AUC,Precisão,Recall,F1-score,Fraudes previstas
Logística padrão,0.587,0.062,0.0000,0.0000,0.0000,0.0
Logística balanceada,0.587,0.062,0.0646,0.3722,0.1101,1037.0


A validação confirma que o balanceamento ajuda a identificar fraudes no limiar padrão, mas não melhorou a capacidade de ordenar os casos por risco: ambos os modelos ficaram com ROC-AUC de 0,587 e PR-AUC de 0,062.

## Etapa 2 — testar Random Forest

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

modelo_rf_validacao = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="most_frequent")),
    ("modelo", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

modelo_rf_validacao.fit(X_dev_fatores, y_dev)

prob_rf_val = modelo_rf_validacao.predict_proba(
    X_val_fatores
)[:, 1]

pred_rf_val = (prob_rf_val >= 0.50).astype(int)

resultado_rf = pd.DataFrame({
    "Random Forest": {
        "ROC-AUC": roc_auc_score(y_val, prob_rf_val),
        "PR-AUC": average_precision_score(y_val, prob_rf_val),
        "Precisão": precision_score(
            y_val, pred_rf_val, zero_division=0
        ),
        "Recall": recall_score(
            y_val, pred_rf_val, zero_division=0
        ),
        "F1-score": f1_score(
            y_val, pred_rf_val, zero_division=0
        ),
        "Fraudes previstas": int(pred_rf_val.sum())
    }
}).T

display(resultado_rf.round(4))

,ROC-AUC,PR-AUC,Precisão,Recall,F1-score,Fraudes previstas
Random Forest,0.5785,0.0586,0.0639,0.3556,0.1084,1001.0


Por enquanto, a regressão logística balanceada é a melhor referência. Os resultados continuam modestos, então não devemos afirmar que já temos um bom detector de fraudes.

## Próximo passo: investigar o limiar na validação
Em vez de mudar novamente o algoritmo, vamos medir como diferentes limiares afetam precisão, recall e quantidade de alertas. Usaremos somente a validação, não o conjunto de teste final.

In [17]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

limiares = [0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.50]

prob_logistica_val = modelo_balanceado.predict_proba(
    X_val_fatores
)[:, 1]

linhas = []

for limiar in limiares:
    previsoes = (prob_logistica_val >= limiar).astype(int)

    linhas.append({
        "Limiar": limiar,
        "Casos sinalizados": int(previsoes.sum()),
        "Precisão": precision_score(
            y_val, previsoes, zero_division=0
        ),
        "Recall": recall_score(
            y_val, previsoes, zero_division=0
        ),
        "F1-score": f1_score(
            y_val, previsoes, zero_division=0
        )
    })

tabela_limites = pd.DataFrame(linhas)
display(tabela_limites.round(4))

,Limiar,Casos sinalizados,Precisão,Recall,F1-score
0,0.05,4000,0.0450,1.0000,0.0861
1,0.08,4000,0.0450,1.0000,0.0861
2,0.10,4000,0.0450,1.0000,0.0861
3,0.15,4000,0.0450,1.0000,0.0861
4,0.20,4000,0.0450,1.0000,0.0861
5,0.30,4000,0.0450,1.0000,0.0861
6,0.50,1037,0.0646,0.3722,0.1101


encontramos um detalhe importante: os resultados mostram que, para a regressão logística balanceada, todos os limiares de 5% a 30% classificaram os 4.000 casos como fraude. Isso acontece porque o balanceamento deslocou as probabilidades estimadas para cima.
Mas há um ponto: não devemos escolher o limiar apenas pelo F1-score. Precisamos considerar quantos alertas a equipe consegue investigar e quanto custa deixar uma fraude passar.

In [18]:
print("Resumo das probabilidades na validação:")
display(
    pd.Series(prob_logistica_val)
    .describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
    .round(4)
)

print("\nMenores probabilidades:")
print(prob_logistica_val.min().round(4))

print("\nMaiores probabilidades:")
print(prob_logistica_val.max().round(4))

Resumo das probabilidades na validação:


count    4000.0000
mean        0.4863
std         0.0778
min         0.3834
50%         0.4871
75%         0.5139
90%         0.5962
95%         0.6215
99%         0.6928
max         0.7785
dtype: float64


Menores probabilidades:
0.3834

Maiores probabilidades:
0.7785


In [19]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd

limiares = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75]

linhas = []

for limiar in limiares:
    previsoes = (prob_logistica_val >= limiar).astype(int)

    linhas.append({
        "Limiar": limiar,
        "Casos sinalizados": int(previsoes.sum()),
        "Precisão": precision_score(
            y_val, previsoes, zero_division=0
        ),
        "Recall": recall_score(
            y_val, previsoes, zero_division=0
        ),
        "F1-score": f1_score(
            y_val, previsoes, zero_division=0
        )
    })

display(pd.DataFrame(linhas).round(4))

,Limiar,Casos sinalizados,Precisão,Recall,F1-score
0,0.40,3019,0.0494,0.8278,0.0932
1,0.45,3019,0.0494,0.8278,0.0932
2,0.50,1037,0.0646,0.3722,0.1101
3,0.55,889,0.0709,0.3500,0.1179
4,0.60,382,0.0838,0.1778,0.1139
5,0.65,117,0.0855,0.0556,0.0673
6,0.70,18,0.1111,0.0111,0.0202
7,0.75,6,0.0000,0.0000,0.0000


O limiar de 0,55 reduz os alertas em relação a 0,50 e melhora um pouco a precisão e o F1-score, com uma pequena queda no recall.
Mesmo assim, 7,09% de precisão significa que aproximadamente 93% dos alertas são falsos positivos. Isso ainda é fraco para uma operação real.

## melhorar a calibração e a avaliação
Antes de escolher definitivamente o limiar, precisamos verificar se as probabilidades do modelo balanceado representam adequadamente o risco. O balanceamento de classes pode alterar essas probabilidades.
O próximo experimento será usar a regressão logística sem pesos, que tende a produzir probabilidades menos deslocadas pela ponderação das classes, e comparar seu desempenho na mesma validação.
Execute esta célula:

In [20]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

modelo_sem_pesos_fatores = Pipeline(steps=[
    ("imputador", SimpleImputer(strategy="most_frequent")),
    ("modelo", LogisticRegression(max_iter=1000))
])

modelo_sem_pesos_fatores.fit(X_dev_fatores, y_dev)

prob_sem_pesos_val = modelo_sem_pesos_fatores.predict_proba(
    X_val_fatores
)[:, 1]

comparacao_probabilidades = pd.DataFrame({
    "Modelo balanceado": {
        "ROC-AUC": roc_auc_score(y_val, prob_logistica_val),
        "PR-AUC": average_precision_score(y_val, prob_logistica_val),
        "Brier Score": brier_score_loss(y_val, prob_logistica_val),
        "Probabilidade média": prob_logistica_val.mean()
    },
    "Modelo sem pesos": {
        "ROC-AUC": roc_auc_score(y_val, prob_sem_pesos_val),
        "PR-AUC": average_precision_score(y_val, prob_sem_pesos_val),
        "Brier Score": brier_score_loss(y_val, prob_sem_pesos_val),
        "Probabilidade média": prob_sem_pesos_val.mean()
    }
}).T

display(comparacao_probabilidades.round(4))
print("Prevalência real na validação:", round(y_val.mean(), 4))

,ROC-AUC,PR-AUC,Brier Score,Probabilidade média
Modelo balanceado,0.587,0.062,0.2415,0.4863
Modelo sem pesos,0.587,0.062,0.0428,0.0449


Prevalência real na validação: 0.045


O modelo sem pesos é a melhor escolha provisória para continuar:
- A ROC-AUC e a PR-AUC são iguais, então não há evidência de que o balanceamento melhore a ordenação dos casos.
- A probabilidade média do modelo sem pesos está próxima da prevalência real.
- O Brier Score é muito menor, o que indica menor erro probabilístico global nesta validação.
Uma ressalva: probabilidade média próxima da prevalência não comprova, sozinha, que as probabilidades estejam perfeitamente calibradas. Para isso, também precisamos analisar a calibração por faixas de risco.

## Próximo passo: escolher um limiar usando o modelo sem pesos
Vamos testar os limiares na validação e observar o equilíbrio entre precisão, recall e quantidade de alertas. Não vamos usar o conjunto de teste para escolher o limiar.

In [21]:
from sklearn.metrics import precision_score, recall_score, f1_score

limiares = [0.03, 0.04, 0.05, 0.06, 0.08, 0.10, 0.15]

linhas = []

for limiar in limiares:
    previsoes = (prob_sem_pesos_val >= limiar).astype(int)

    linhas.append({
        "Limiar": limiar,
        "Casos sinalizados": int(previsoes.sum()),
        "Precisão": precision_score(
            y_val, previsoes, zero_division=0
        ),
        "Recall": recall_score(
            y_val, previsoes, zero_division=0
        ),
        "F1-score": f1_score(
            y_val, previsoes, zero_division=0
        )
    })

display(pd.DataFrame(linhas).round(4))

,Limiar,Casos sinalizados,Precisão,Recall,F1-score
0,0.03,3019,0.0494,0.8278,0.0932
1,0.04,2750,0.0516,0.7889,0.0969
2,0.05,889,0.0709,0.3500,0.1179
3,0.06,439,0.0843,0.2056,0.1195
4,0.08,122,0.0820,0.0556,0.0662
5,0.10,22,0.1818,0.0222,0.0396
6,0.15,5,0.0000,0.0000,0.0000


O que isso significa?
- Precisão de 8,43%: aproximadamente 8 em cada 100 sinistros sinalizados seriam fraudes confirmadas nesse conjunto de validação.
- Recall de 20,56%: o modelo identifica aproximadamente 21 em cada 100 fraudes existentes.
- 439 casos sinalizados: a equipe de investigação precisaria analisar esses casos, em vez de todos os 4.000 registros da validação.
O limiar 0,06 reduz o volume de alertas, mas também deixa passar mais fraudes. A escolha depende do custo de investigar um caso e do prejuízo de deixar uma fraude passar.
Um detalhe importante: como a prevalência de fraude é de aproximadamente 4,5%, a precisão de 8,43% é superior à proporção de fraudes na base, mas o ganho ainda é limitado. Precisamos avaliar se conseguimos melhorar a ordenação dos casos de maior risco

## investigar a capacidade de separar fraudes
Antes de trocar de algoritmo, vamos verificar se os cinco indicadores que criamos realmente ajudam a distinguir os sinistros fraudulentos

In [22]:

from sklearn.metrics import average_precision_score, roc_auc_score

print("Desempenho na validação")
print(f"ROC-AUC: {roc_auc_score(y_val, prob_sem_pesos_val):.4f}")
print(f"PR-AUC:  {average_precision_score(y_val, prob_sem_pesos_val):.4f}")
print(f"Prevalência de fraude: {y_val.mean():.4f}")

print("\nDistribuição das probabilidades previstas")
display(
    pd.Series(prob_sem_pesos_val, name="Probabilidade prevista")
    .describe()
    .round(4)
)


Desempenho na validação
ROC-AUC: 0.5870
PR-AUC:  0.0620
Prevalência de fraude: 0.0450

Distribuição das probabilidades previstas


count    4000.0000
mean        0.0449
std         0.0156
min         0.0285
25%         0.0388
50%         0.0429
75%         0.0472
max         0.1516
Name: Probabilidade prevista, dtype: float64

os resultados mostram que o modelo está aprendendo algum sinal, mas a capacidade de separar fraudes de não fraudes ainda é baixa

# VAMOS USAR A BASE TRATADA

In [23]:

from pathlib import Path
import pandas as pd

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

CAMINHO_TRATADO = (
    ROOT_DIR / "data" / "processed"
    / "sinistros_seguros_tratados.csv"
)

df_tratado_modelagem = pd.read_csv(CAMINHO_TRATADO)

print("Dimensões da base tratada:", df_tratado_modelagem.shape)
print("\nColunas:")
print(df_tratado_modelagem.columns.tolist())

print("\nValores ausentes por coluna:")
display(df_tratado_modelagem.isna().sum())

print("\nDistribuição do alvo:")
display(
    df_tratado_modelagem["fraude_confirmada"]
    .value_counts()
    .rename(index={0: "Não fraude", 1: "Fraude"})
)


Dimensões da base tratada: (20000, 14)

Colunas:
['id_sinistro', 'idade_segurado', 'tempo_cliente_anos', 'tipo_seguro', 'regiao', 'valor_sinistro', 'valor_apolice', 'sinistros_anteriores', 'dias_desde_contratacao', 'tipo_ocorrencia', 'canal_aviso', 'hora_ocorrencia', 'dias_para_comunicar', 'fraude_confirmada']

Valores ausentes por coluna:


id_sinistro               0
idade_segurado            0
tempo_cliente_anos        0
tipo_seguro               0
regiao                    0
valor_sinistro            0
valor_apolice             0
sinistros_anteriores      0
dias_desde_contratacao    0
tipo_ocorrencia           0
canal_aviso               0
hora_ocorrencia           0
dias_para_comunicar       0
fraude_confirmada         0
dtype: int64


Distribuição do alvo:


fraude_confirmada
Não fraude    19097
Fraude          903
Name: count, dtype: int64

In [24]:

from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

CAMINHO_TRATADO = (
    ROOT_DIR / "data" / "processed"
    / "sinistros_seguros_tratados.csv"
)

# Carregar a base tratada
df_modelagem = pd.read_csv(CAMINHO_TRATADO)

# Separar variáveis explicativas e alvo
X = df_modelagem.drop(
    columns=["fraude_confirmada", "id_sinistro"]
)
y = df_modelagem["fraude_confirmada"]

# Separar treino e teste, preservando a proporção de fraudes
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Base utilizada:", CAMINHO_TRATADO.name)
print("Total de registros:", len(df_modelagem))
print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

print(f"\nFraudes no treino: {y_train.sum()} de {len(y_train)}")
print(f"Proporção de fraudes no treino: {y_train.mean():.2%}")

print(f"\nFraudes no teste: {y_test.sum()} de {len(y_test)}")
print(f"Proporção de fraudes no teste: {y_test.mean():.2%}")

print("\nValores ausentes em X:", X.isna().sum().sum())
print("Duplicatas completas:", df_modelagem.duplicated().sum())


Base utilizada: sinistros_seguros_tratados.csv
Total de registros: 20000
Treino: (16000, 12)
Teste: (4000, 12)

Fraudes no treino: 722 de 16000
Proporção de fraudes no treino: 4.51%

Fraudes no teste: 181 de 4000
Proporção de fraudes no teste: 4.52%

Valores ausentes em X: 0
Duplicatas completas: 0


In [25]:

print("Tipos das variáveis:")
display(X.dtypes.to_frame("Tipo"))

print("\nResumo das variáveis numéricas:")
display(X.describe().T.round(2))

print("\nCategorias das variáveis de texto:")
for coluna in X.select_dtypes(include=["object", "category"]).columns:
    print(f"\n{coluna}:")
    print(X[coluna].value_counts().head(10))

print("\nDistribuição do alvo:")
display(
    y.value_counts()
    .rename(index={0: "Não fraude", 1: "Fraude"})
    .to_frame("Quantidade")
)


Tipos das variáveis:


,Tipo
idade_segurado,float64
tempo_cliente_anos,int64
tipo_seguro,str
regiao,str
valor_sinistro,float64
valor_apolice,float64
sinistros_anteriores,int64
dias_desde_contratacao,int64
tipo_ocorrencia,str
canal_aviso,str



Resumo das variáveis numéricas:


,count,mean,std,min,25%,50%,75%,max
idade_segurado,20000.0,50.78,19.01,18.00,35.00,51.00,67.00,84.00
tempo_cliente_anos,20000.0,12.04,7.23,0.00,6.00,12.00,18.00,24.00
valor_sinistro,20000.0,8062.38,10897.68,98.81,2575.54,4904.76,9377.59,322995.08
valor_apolice,20000.0,10321.67,8323.35,532.88,5011.72,7992.14,12958.46,135922.86
sinistros_anteriores,20000.0,0.70,0.84,0.00,0.00,1.00,1.00,6.00
dias_desde_contratacao,20000.0,1822.47,1046.86,1.00,917.00,1818.00,2720.00,3649.00
hora_ocorrencia,20000.0,11.51,6.93,0.00,5.00,12.00,18.00,23.00
dias_para_comunicar,20000.0,14.93,8.92,0.00,7.00,15.00,23.00,30.00



Categorias das variáveis de texto:

tipo_seguro:
tipo_seguro
automóvel      12968
residencial     3062
vida            1990
empresarial     1980
Name: count, dtype: int64

regiao:
regiao
Sudeste         4050
Norte           4021
Nordeste        4010
Sul             3977
Centro-Oeste    3942
Name: count, dtype: int64

tipo_ocorrencia:
tipo_ocorrencia
Dano climático    3943
Outros            3926
Colisão           3894
Incêndio          3888
Roubo             3749
Não informado      600
Name: count, dtype: int64

canal_aviso:
canal_aviso
Aplicativo       4909
Site             4900
Telefone         4822
Corretor         4769
Não informado     600
Name: count, dtype: int64

Distribuição do alvo:


C:\Users\vanio\AppData\Local\Temp\ipykernel_9928\2580986623.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for coluna in X.select_dtypes(include=["object", "category"]).columns:


,Quantidade
fraude_confirmada,
Não fraude,19097
Fraude,903


In [26]:

# Comparar as médias das variáveis numéricas entre os dois grupos
colunas_numericas = X.select_dtypes(include="number").columns

resumo = X.assign(fraude_confirmada=y).groupby(
    "fraude_confirmada"
)[colunas_numericas].mean().T

resumo.columns = ["Não fraude", "Fraude"]
resumo["Diferença percentual"] = (
    (resumo["Fraude"] - resumo["Não fraude"])
    / resumo["Não fraude"].replace(0, float("nan"))
) * 100

display(resumo.round(2))


,Não fraude,Fraude,Diferença percentual
idade_segurado,50.81,50.11,-1.38
tempo_cliente_anos,12.05,11.83,-1.84
valor_sinistro,7987.57,9644.50,20.74
valor_apolice,10328.71,10172.63,-1.51
sinistros_anteriores,0.70,0.74,5.83
dias_desde_contratacao,1822.54,1821.06,-0.08
hora_ocorrencia,11.54,11.04,-4.30
dias_para_comunicar,14.87,16.27,9.41


Variável	Diferença entre os grupos	Interpretação inicial
valor_sinistro	+20,74%	Pista potencialmente útil
dias_para_comunicar	+9,41%	Pista potencialmente útil
sinistros_anteriores	+5,83%	Diferença pequena
hora_ocorrencia	−4,30%	Diferença pequena
dias_desde_contratacao	−0,08%	Médias praticamente iguais
idade_segurado	−1,38%	Médias semelhantes
tempo_cliente_anos	−1,84%	Médias semelhantes

In [27]:

colunas_categoricas = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for coluna in colunas_categoricas:
    print(f"\n{'=' * 55}")
    print(f"Variável: {coluna}")

    tabela = pd.crosstab(
        df_modelagem[coluna],
        df_modelagem["fraude_confirmada"],
        normalize="index"
    ) * 100

    tabela = tabela.rename(columns={
        0: "Não fraude (%)",
        1: "Fraude (%)"
    })

    print(tabela.round(2))



Variável: tipo_seguro
fraude_confirmada  Não fraude (%)  Fraude (%)
tipo_seguro                                  
automóvel                   95.34        4.66
empresarial                 95.76        4.24
residencial                 95.82        4.18
vida                        95.63        4.37

Variável: regiao
fraude_confirmada  Não fraude (%)  Fraude (%)
regiao                                       
Centro-Oeste                95.10        4.90
Nordeste                    95.56        4.44
Norte                       95.67        4.33
Sudeste                     95.53        4.47
Sul                         95.55        4.45

Variável: tipo_ocorrencia
fraude_confirmada  Não fraude (%)  Fraude (%)
tipo_ocorrencia                              
Colisão                     95.22        4.78
Dano climático              95.46        4.54
Incêndio                    95.58        4.42
Não informado               96.67        3.33
Outros                      95.54        4.46
Roubo       

C:\Users\vanio\AppData\Local\Temp\ipykernel_9928\540266767.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_categoricas = X.select_dtypes(


1. O que os dados nos dizem
- tipo_seguro: pouca variação nas taxas de fraude.
- regiao: o Centro-Oeste apresenta 4,90%, mas precisamos considerar a quantidade de registros e a incerteza dessa diferença.
- tipo_ocorrencia e canal_aviso: diferenças modestas, sem uma separação evidente entre fraude e não fraude.
Esses resultados não provam que as categorias são inúteis. Apenas sugerem que, isoladamente, elas têm pouco poder discriminativo aparente.

## Nova estratégia: criar um modelo de referência com todas as variáveis
Vamos comparar um modelo que use as variáveis numéricas e categóricas originais, com pré-processamento apropriado. Assim, não descartamos informações potencialmente úteis.

In [28]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# Identificar os tipos de variáveis
colunas_numericas = X_train.select_dtypes(include="number").columns.tolist()
colunas_categoricas = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# Preparar variáveis numéricas
preprocessamento_numerico = Pipeline([
    ("imputador", SimpleImputer(strategy="median")),
    ("escala", StandardScaler())
])

# Preparar variáveis categóricas
preprocessamento_categorico = Pipeline([
    ("imputador", SimpleImputer(strategy="most_frequent")),
    ("codificador", OneHotEncoder(handle_unknown="ignore"))
])

preprocessador_completo = ColumnTransformer([
    ("numericas", preprocessamento_numerico, colunas_numericas),
    ("categoricas", preprocessamento_categorico, colunas_categoricas)
])

modelo_referencia = Pipeline([
    ("preprocessamento", preprocessador_completo),
    ("classificador", LogisticRegression(
        max_iter=1000,
        class_weight=None
    ))
])

# Treinar somente no conjunto de desenvolvimento
modelo_referencia.fit(X_dev, y_dev)

# Avaliar na validação
prob_referencia_val = modelo_referencia.predict_proba(X_val)[:, 1]

print("Modelo de referência — validação")
print(f"ROC-AUC: {roc_auc_score(y_val, prob_referencia_val):.4f}")
print(f"PR-AUC:  {average_precision_score(y_val, prob_referencia_val):.4f}")
print(f"Prevalência: {y_val.mean():.4f}")


Modelo de referência — validação
ROC-AUC: 0.5198
PR-AUC:  0.0539
Prevalência: 0.0450


C:\Users\vanio\AppData\Local\Temp\ipykernel_9928\1290768765.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_categoricas = X_train.select_dtypes(


O modelo com cinco indicadores continua sendo o melhor dos dois experimentos. Porém, seu desempenho ainda é limitado.

In [32]:

from sklearn.metrics import roc_auc_score, average_precision_score

modelos = {
    "Logística - 5 indicadores": (
        modelo_sem_pesos_fatores,
        X_val_fatores
    ),
    "Logística - variáveis originais": (
        modelo_referencia,
        X_val
    ),
}

resultados = []

for nome, (modelo, dados_validacao) in modelos.items():
    probabilidades = modelo.predict_proba(dados_validacao)[:, 1]

    resultados.append({
        "Modelo": nome,
        "ROC-AUC": roc_auc_score(y_val, probabilidades),
        "PR-AUC": average_precision_score(y_val, probabilidades)
    })

tabela_resultados = pd.DataFrame(resultados)
display(tabela_resultados.round(4))


,Modelo,ROC-AUC,PR-AUC
0,Logística - 5 indicadores,0.5870,0.0620
1,Logística - variáveis originais,0.5198,0.0539


In [33]:

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd

# 1. Random Forest
modelo_rf_fatores = Pipeline([
    ("imputador", SimpleImputer(strategy="most_frequent")),
    ("modelo", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# 2. Gradient Boosting
modelo_gb_fatores = Pipeline([
    ("imputador", SimpleImputer(strategy="most_frequent")),
    ("modelo", HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=42
    ))
])

# Treinar somente com os dados de desenvolvimento
modelo_rf_fatores.fit(X_dev_fatores, y_dev)
modelo_gb_fatores.fit(X_dev_fatores, y_dev)

# Probabilidades na validação
prob_rf_val = modelo_rf_fatores.predict_proba(X_val_fatores)[:, 1]
prob_gb_val = modelo_gb_fatores.predict_proba(X_val_fatores)[:, 1]

# Comparação
resultados_novos = pd.DataFrame([
    {
        "Modelo": "Random Forest",
        "ROC-AUC": roc_auc_score(y_val, prob_rf_val),
        "PR-AUC": average_precision_score(y_val, prob_rf_val)
    },
    {
        "Modelo": "Gradient Boosting",
        "ROC-AUC": roc_auc_score(y_val, prob_gb_val),
        "PR-AUC": average_precision_score(y_val, prob_gb_val)
    }
])

display(resultados_novos.round(4))


,Modelo,ROC-AUC,PR-AUC
0,Random Forest,0.5785,0.0586
1,Gradient Boosting,0.5751,0.0591


# Resultado da comparação
Modelo	ROC-AUC	PR-AUC
Regressão Logística — 5 indicadores	0,5870	0,0620
Random Forest	0,5785	0,0586
Gradient Boosting	0,5751	0,0591
Regressão Logística — variáveis originais	0,5198	0,0539

In [34]:

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Limiar escolhido na etapa de validação
limiar = 0.06

# Probabilidades previstas pelo modelo
probabilidades = prob_sem_pesos_val

# Converter probabilidades em classificações
previsoes = (probabilidades >= limiar).astype(int)

# Calcular a matriz de confusão
matriz = confusion_matrix(y_val, previsoes)
tn, fp, fn, tp = matriz.ravel()

print("AVALIAÇÃO OPERACIONAL — VALIDAÇÃO")
print("-" * 45)
print(f"Limiar utilizado: {limiar:.2f}")
print(f"Total de sinistros: {len(y_val)}")
print(f"Casos sinalizados: {previsoes.sum()}")
print(f"Fraudes identificadas: {tp}")
print(f"Falsos alertas: {fp}")
print(f"Fraudes não identificadas: {fn}")
print(f"Casos corretamente não sinalizados: {tn}")
print(f"Precisão: {precision_score(y_val, previsoes, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_val, previsoes, zero_division=0):.4f}")
print(f"F1-score: {f1_score(y_val, previsoes, zero_division=0):.4f}")


AVALIAÇÃO OPERACIONAL — VALIDAÇÃO
---------------------------------------------
Limiar utilizado: 0.06
Total de sinistros: 4000
Casos sinalizados: 439
Fraudes identificadas: 37
Falsos alertas: 402
Fraudes não identificadas: 143
Casos corretamente não sinalizados: 3418
Precisão: 0.0843
Recall: 0.2056
F1-score: 0.1195


In [35]:

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Criar os cinco indicadores usando o limite calculado no desenvolvimento
X_test_fatores = criar_fatores(X_test, limite_valor_dev)

# Gerar probabilidades com a Regressão Logística escolhida
probabilidades_teste = modelo_sem_pesos_fatores.predict_proba(
    X_test_fatores
)[:, 1]

# Aplicar o limiar definido na validação
limiar = 0.06
previsoes_teste = (probabilidades_teste >= limiar).astype(int)

# Calcular métricas
matriz = confusion_matrix(y_test, previsoes_teste)
tn, fp, fn, tp = matriz.ravel()

print("AVALIAÇÃO FINAL — CONJUNTO DE TESTE")
print("=" * 45)
print(f"Total de sinistros: {len(y_test)}")
print(f"Fraudes reais: {int(y_test.sum())}")
print(f"Casos sinalizados: {int(previsoes_teste.sum())}")
print(f"Fraudes identificadas: {tp}")
print(f"Falsos alertas: {fp}")
print(f"Fraudes não identificadas: {fn}")
print(f"Casos corretamente não sinalizados: {tn}")
print("-" * 45)
print(f"ROC-AUC: {roc_auc_score(y_test, probabilidades_teste):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, probabilidades_teste):.4f}")
print(f"Precisão: {precision_score(y_test, previsoes_teste, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, previsoes_teste, zero_division=0):.4f}")
print(f"F1-score: {f1_score(y_test, previsoes_teste, zero_division=0):.4f}")

print("\nMatriz de confusão:")
print(matriz)


AVALIAÇÃO FINAL — CONJUNTO DE TESTE
Total de sinistros: 4000
Fraudes reais: 181
Casos sinalizados: 429
Fraudes identificadas: 31
Falsos alertas: 398
Fraudes não identificadas: 150
Casos corretamente não sinalizados: 3421
---------------------------------------------
ROC-AUC: 0.5987
PR-AUC: 0.0638
Precisão: 0.0723
Recall: 0.1713
F1-score: 0.1016

Matriz de confusão:
[[3421  398]
 [ 150   31]]


In [36]:

from pathlib import Path
import json
import joblib

# Criar a pasta de artefatos, caso ainda não exista
pasta_artifacts = Path("artifacts")
pasta_artifacts.mkdir(parents=True, exist_ok=True)

# Conferir se o modelo escolhido existe na memória
print("Modelo disponível:", "modelo_sem_pesos_fatores" in globals())
print("Limite de valor disponível:", "limite_valor_dev" in globals())
print("Função de fatores disponível:", "criar_fatores" in globals())


Modelo disponível: True
Limite de valor disponível: True
Função de fatores disponível: True


In [37]:

from pathlib import Path
import joblib
import json

pasta_artifacts = Path("artifacts")
pasta_artifacts.mkdir(parents=True, exist_ok=True)

# Salvar o modelo treinado
joblib.dump(
    modelo_sem_pesos_fatores,
    pasta_artifacts / "modelo_fraudes.joblib"
)

# Salvar os parâmetros necessários para reproduzir as previsões
metadados = {
    "modelo": "Regressão Logística",
    "features": [
        "sinistros_3_ou_mais",
        "contratacao_recente",
        "sinistro_valor_alto",
        "comunicacao_tardia",
        "ocorrencia_madrugada"
    ],
    "limite_valor_dev": float(limite_valor_dev),
    "limiar_classificacao": 0.06,
    "roc_auc_teste": 0.5987,
    "pr_auc_teste": 0.0638,
    "precisao_teste": 0.0723,
    "recall_teste": 0.1713,
    "f1_teste": 0.1016,
    "observacao": (
        "Modelo experimental treinado com dados sintéticos. "
        "Resultados de teste documentados para referência."
    )
}

with open(
    pasta_artifacts / "metadados_modelo.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(metadados, arquivo, ensure_ascii=False, indent=4)

print("Modelo e metadados salvos com sucesso!")
print("Arquivos:", [arquivo.name for arquivo in pasta_artifacts.iterdir()])


Modelo e metadados salvos com sucesso!
Arquivos: ['metadados_modelo.json', 'modelo_fraudes.joblib']


In [38]:

import importlib.util

if importlib.util.find_spec("mlflow") is not None:
    print("MLflow já está instalado.")
else:
    print("MLflow ainda não está instalado.")


MLflow já está instalado.


In [39]:

import mlflow
import mlflow.sklearn
import json
from pathlib import Path

# Configurar onde o MLflow armazenará os registros
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("prevencao_fraude_seguro")

# Ler os metadados já salvos
with open("artifacts/metadados_modelo.json", "r", encoding="utf-8") as arquivo:
    metadados = json.load(arquivo)

with mlflow.start_run(run_name="regressao_logistica_fraudes"):
    # Registrar parâmetros do modelo
    mlflow.log_param("modelo", metadados["modelo"])
    mlflow.log_param("limiar_classificacao", metadados["limiar_classificacao"])
    mlflow.log_param("limite_valor_dev", metadados["limite_valor_dev"])
    mlflow.log_param("quantidade_features", len(metadados["features"]))
    mlflow.log_param("origem_dados", "sinteticos")

    # Registrar métricas da avaliação final já realizada
    mlflow.log_metric("roc_auc_teste", metadados["roc_auc_teste"])
    mlflow.log_metric("pr_auc_teste", metadados["pr_auc_teste"])
    mlflow.log_metric("precisao_teste", metadados["precisao_teste"])
    mlflow.log_metric("recall_teste", metadados["recall_teste"])
    mlflow.log_metric("f1_teste", metadados["f1_teste"])

    # Registrar o modelo salvo, sem treiná-lo novamente
    mlflow.sklearn.log_model(
        sk_model=joblib.load("artifacts/modelo_fraudes.joblib"),
        name="modelo_fraudes"
    )

print("Registro no MLflow concluído!")
print("Experimento: prevencao_fraude_seguro")


C:\Users\vanio\Documents\MEUS_PROJETOS\prevencao_fraude_seguro\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MlflowException: The filesystem tracking backend (e.g., './mlruns') is in maintenance mode and will not receive further updates. Please migrate to a database backend (e.g., 'sqlite:///mlflow.db') to access the latest MLflow features. The `mlflow migrate-filestore` tool migrates your existing data losslessly. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance. If the filesystem backend is required for your workflow, set `MLFLOW_ALLOW_FILE_STORE=true` to opt out of this exception.

In [40]:

import mlflow
import joblib
import json
from pathlib import Path

# Usar SQLite para armazenar o histórico do MLflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("prevencao_fraude_seguro")

# Ler os metadados do modelo já salvo
with open("artifacts/metadados_modelo.json", "r", encoding="utf-8") as arquivo:
    metadados = json.load(arquivo)

# Registrar o experimento sem treinar novamente
with mlflow.start_run(run_name="regressao_logistica_fraudes"):
    mlflow.log_param("modelo", metadados["modelo"])
    mlflow.log_param("limiar_classificacao", metadados["limiar_classificacao"])
    mlflow.log_param("limite_valor_dev", metadados["limite_valor_dev"])
    mlflow.log_param("quantidade_features", len(metadados["features"]))
    mlflow.log_param("origem_dados", "sinteticos")

    mlflow.log_metrics({
        "roc_auc_teste": metadados["roc_auc_teste"],
        "pr_auc_teste": metadados["pr_auc_teste"],
        "precisao_teste": metadados["precisao_teste"],
        "recall_teste": metadados["recall_teste"],
        "f1_teste": metadados["f1_teste"]
    })

    modelo_salvo = joblib.load("artifacts/modelo_fraudes.joblib")

    mlflow.sklearn.log_model(
        sk_model=modelo_salvo,
        name="modelo_fraudes"
    )

print("Experimento registrado com sucesso!")
print("Armazenamento: SQLite")
print("Experimento: prevencao_fraude_seguro")


2026/10/14 11:49:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/14 11:49:21 INFO mlflow.store.db.utils: Updating database tables
2026/10/14 11:49:23 INFO mlflow.tracking.fluent: Experiment with name 'prevencao_fraude_seguro' does not exist. Creating a new experiment.


MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['numpy.dtype'].

In [41]:

import mlflow
import json

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("prevencao_fraude_seguro")

with open("artifacts/metadados_modelo.json", "r", encoding="utf-8") as arquivo:
    metadados = json.load(arquivo)

with mlflow.start_run(run_name="regressao_logistica_fraudes"):
    mlflow.log_params({
        "modelo": metadados["modelo"],
        "limiar_classificacao": metadados["limiar_classificacao"],
        "limite_valor_dev": metadados["limite_valor_dev"],
        "quantidade_features": len(metadados["features"]),
        "origem_dados": "sinteticos"
    })

    mlflow.log_metrics({
        "roc_auc_teste": metadados["roc_auc_teste"],
        "pr_auc_teste": metadados["pr_auc_teste"],
        "precisao_teste": metadados["precisao_teste"],
        "recall_teste": metadados["recall_teste"],
        "f1_teste": metadados["f1_teste"]
    })

print("Parâmetros e métricas registrados no MLflow!")


Parâmetros e métricas registrados no MLflow!
